In [ ]:
from nltk.corpus import wordnet as wn
from collections import defaultdict

def table_to_propositions(table):
    """
    Convert a table (list of dicts) into semantic propositions.
    Example: 
      Table row {"Name": "Alice", "Age": "30"} → {("Name", "Alice"), ("Age", "30")}
    """
    propositions = set()
    for row in table:
        for header, value in row.items():
            # Normalize values and split into tokens for synonym handling
            propositions.add((header.strip().lower(), value.strip().lower()))
    return propositions

def synonym_check(token1, token2):
    """Check if two tokens are synonyms using WordNet."""
    synsets1 = wn.synsets(token1)
    synsets2 = wn.synsets(token2)
    return len(synsets1) > 0 and len(synsets2) > 0 and synsets1[0] == synsets2[0]

def spice_score(reference, candidate):
    """
    Compute SPICE-like F-score between reference and candidate tables.
    """
    ref_props = table_to_propositions(reference)
    cand_props = table_to_propositions(candidate)
    
    # Match propositions with exact or synonym matches
    match_counts = defaultdict(int)
    for (h_cand, v_cand) in cand_props:
        for (h_ref, v_ref) in ref_props:
            if h_cand == h_ref:
                if v_cand == v_ref or synonym_check(v_cand, v_ref):
                    match_counts[(h_cand, v_cand)] += 1
    
    tp = sum(min(count, 1) for count in match_counts.values())  # True positives
    fp = len(cand_props) - tp                                   # False positives
    fn = len(ref_props) - tp                                    # False negatives
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {"SPICE-F1": f1, "Precision": precision, "Recall": recall}

# Example Usage
reference_table = [
    {"Name": "Alice", "Age": "30"},
    {"Name": "Bob", "Age": "25"}
]

candidate_table = [
    {"Name": "Alice", "Age": "30"},
    {"Name": "Charlie", "Age": "25"}
]

score = spice_score(reference_table, candidate_table)
print(score)  # Output: {'SPICE-F1': 0.75, 'Precision': 0.75, 'Recall': 0.75}

: 